# Group results: Figures 2–3 and Tables S1–S8

*Edited by [Ali Ekhlasi](https://github.com/aliekhlasi), Postdoctoral Researcher, Department of Cognitive Sciences, University of California, Irvine. License: GNU GPL v3.0 (see `LICENSE`).*

This notebook turns the per-subject searchlight maps into the group results in
*"Is This the Real Life? Is This Just Fantasy? A Domain Adaptation Framework for
Identifying Higher-Order Reality Representations"* (full paper in preparation):

| Output | What it shows |
|---|---|
| Figure 2 a–c | Content decoding: within domain, across domain, after DA alignment |
| Figure 2 d–f | Reality decoding: within content, across content, after DA alignment |
| Figure 3 a–c | Content: reality-independent, reality-alignable, reality-dependent |
| Figure 3 d–f | Reality: content-independent, content-alignable, content-dependent |
| Tables S1–S8 | Harvard–Oxford regions inside each group overlap map |

**Inputs.** The MNI-space maps written by `run_searchlight_pipeline.sh` (searchlight →
Cohen's d → warp to MNI), for 18 subjects, 10 iterations, 12 mm spheres, Subspace
Alignment (SA). The expected layout is

```
<RESULTS_ROOT>/Cross_Domain_Content_Decoding/{P2P,I2I,P2I,I2P}/<subject>/...
<RESULTS_ROOT>/Cross_Content_Domain_Decoding/{Ci2Ci,Cj2Cj,Ci2Cj,Cj2Ci}/<subject>/...
```

Set `RESULTS_ROOT` in the next cell, or export the environment variable
`CCN_RESULTS_ROOT` before starting Jupyter. All outputs go to `OUTPUT_ROOT`.

**Naming.** Content decoding: P = perception, I = imagery, so `P2I` means train on
perception and test on imagery. Reality decoding: Ci and Cj are the two object
categories (dogs, scissors), so `Ci2Cj` means train on one category and test on the
other.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from nilearn import datasets, plotting
from nilearn.image import load_img, new_img_like, resample_to_img

# ---------------------------------------------------------------- paths
RESULTS_ROOT = Path(os.environ.get("CCN_RESULTS_ROOT", "outputs/results/searchlight")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("CCN_OUTPUT_ROOT", "outputs/ccn2026")).expanduser()

# ---------------------------------------------------------------- run settings
SUBJECTS = [str(s) for s in range(1, 19)]  # 18 subjects, folder names 1..18
NITER = 10        # resampling iterations
RADIUS = 12       # searchlight radius (mm)
DA_METHOD = "SA"  # Subspace Alignment

# ---------------------------------------------------------------- thresholds (paper Table 3)
THR = {
    "cd_p2p": 0.70,        # within-domain content decoding, perception
    "cd_i2i": 0.60,        # within-domain content decoding, imagery
    "rd_within": 0.60,     # within-content reality decoding
    "cross": 0.60,         # cross-domain / cross-content decoding (baseline and after DA)
    "dcd_ratio": 1.0,      # DcdRatio = post-DA accuracy / baseline cross accuracy
    "covdist_ratio": 1.0,  # CovDistRatio = post-DA CovDist / baseline CovDist
    "percentile": 75,      # "top 25%" cut for CovDist_base, Cohen's d_acc, Cohen's d_dist
}
OVERLAP_MIN_SUBJECTS = 4  # group maps keep voxels found in MORE than 4 subjects (>= 5 of 18)

# ---------------------------------------------------------------- plotting
DISPLAY_MODE = "lzr"
OVERLAP_CMAP = "YlOrRd"

print("RESULTS_ROOT:", RESULTS_ROOT.resolve())
print("OUTPUT_ROOT :", OUTPUT_ROOT.resolve())
assert (RESULTS_ROOT / "Cross_Domain_Content_Decoding").is_dir(), "RESULTS_ROOT has no Cross_Domain_Content_Decoding folder"
assert (RESULTS_ROOT / "Cross_Content_Domain_Decoding").is_dir(), "RESULTS_ROOT has no Cross_Content_Domain_Decoding folder"

## Loading the per-subject maps

Each subject's maps are loaded on the grid of that subject's first map. Within-content
reality maps (`Ci2Ci`, `Cj2Cj`) are stored once per content in `content_*/`
subfolders, and those are averaged. Every map is checked for being finite and
non-constant when it is loaded, so a missing or empty input fails here instead of
producing an empty group map later.

In [ ]:
SUFFIX_BASE = f"{NITER}iter_{RADIUS}mm_BASELINE_mni"     # baseline (no DA)
SUFFIX_DA = f"{NITER}iter_{RADIUS}mm_{DA_METHOD}_mni"    # after DA
SUFFIX_RATIO = f"{NITER}iter_{RADIUS}mm_{DA_METHOD}BASE_mni"   # DA / baseline ratios
COHEN_TAG = f"{NITER}iter_{RADIUS}mm_{DA_METHOD}BASELINE_mni"  # Cohen's d (DA vs baseline)


def _check(data, path):
    finite = np.isfinite(data)
    if not finite.any():
        raise ValueError(f"No finite values in {path}")
    vals = data[finite]
    if np.all(vals == vals[0]):
        raise ValueError(f"Map is constant ({vals[0]}) in {path}")
    return data


def load_map(path, ref_img=None):
    img = load_img(str(path))
    if ref_img is not None and img.shape != ref_img.shape:
        img = resample_to_img(img, ref_img, interpolation="continuous",
                              force_resample=True, copy_header=True)
    return img, _check(img.get_fdata(), path)


def load_avg(folder, filename, ref_img=None):
    """Load folder/filename, or the mean over folder/content_*/filename."""
    direct = folder / filename
    if direct.exists():
        return load_map(direct, ref_img)
    maps, img = [], ref_img
    for cdir in sorted(folder.glob("content_*")):
        p = cdir / filename
        if p.exists():
            i, d = load_map(p, ref_img)
            img = img if img is not None else i
            maps.append(d)
    if not maps:
        raise FileNotFoundError(f"Missing {filename} in {folder} (and no content_* copies)")
    return img, np.nanmean(np.stack(maps), axis=0)


def load_subject(analysis, subject):
    """Return (ref_img, maps) for one subject.

    Keys are the same for both analyses:
      within_a, within_b   within-domain (content) or within-content (reality) accuracy
      cross_ab, cross_ba   cross accuracy, baseline
      da_ab, da_ba         cross accuracy after DA
      ratio_ab, ratio_ba   DcdRatio (DA / baseline accuracy)
      cov_ab, cov_ba       CovDist, baseline
      covratio_ab, ...     CovDistRatio (DA / baseline)
      dz_ab, dz_ba         Cohen's d of accuracy (DA vs baseline)
      covdz_ab, covdz_ba   Cohen's d of CovDist (DA vs baseline)
    """
    if analysis == "content":
        root = RESULTS_ROOT / "Cross_Domain_Content_Decoding"
        within, cross = ("P2P", "I2I"), ("P2I", "I2P")
    else:
        root = RESULTS_ROOT / "Cross_Content_Domain_Decoding"
        within, cross = ("Ci2Ci", "Cj2Cj"), ("Ci2Cj", "Cj2Ci")

    m = {}
    ref, m["within_a"] = load_avg(root / within[0] / subject, f"map_acc_{SUFFIX_BASE}.nii.gz")
    _, m["within_b"] = load_avg(root / within[1] / subject, f"map_acc_{SUFFIX_BASE}.nii.gz", ref)
    for tag, mode in (("ab", cross[0]), ("ba", cross[1])):
        base_dir = root / mode / subject
        da_dir = base_dir / DA_METHOD
        _, m[f"cross_{tag}"] = load_avg(base_dir, f"map_acc_{SUFFIX_BASE}.nii.gz", ref)
        _, m[f"cov_{tag}"] = load_avg(base_dir, f"map_cov_{SUFFIX_BASE}.nii.gz", ref)
        _, m[f"da_{tag}"] = load_avg(da_dir, f"map_acc_{SUFFIX_DA}.nii.gz", ref)
        _, m[f"ratio_{tag}"] = load_avg(da_dir, f"map_acc_{SUFFIX_RATIO}.nii.gz", ref)
        _, m[f"covratio_{tag}"] = load_avg(da_dir, f"map_cov_{SUFFIX_RATIO}.nii.gz", ref)
        _, m[f"dz_{tag}"] = load_avg(da_dir, f"map_cohen_dz_{COHEN_TAG}.nii.gz", ref)
        _, m[f"covdz_{tag}"] = load_avg(da_dir, f"map_cohen_dz_cov_distance_{COHEN_TAG}.nii.gz", ref)
    return ref, m


def pct(data, q=THR["percentile"]):
    """Per-subject, per-map percentile over all finite voxels."""
    vals = data[np.isfinite(data)]
    return np.nanpercentile(vals, q) if vals.size else np.nan

## Interpretation categories

These are the rules behind each figure panel and table, applied voxel by voxel
within each subject. "Top 25%" thresholds (CovDist_base, Cohen's d) are the 75th
percentile of that subject's map.

| Category | Rule (both directions = P→I and I→P, or Ci→Cj and Cj→Ci) |
|---|---|
| *within* (Fig. 2a / 2d) | within-domain (or within-content) accuracy above threshold in both |
| *ever* (Tables S1 / S5) | the same, in at least one |
| *cross* (Fig. 2b / 2e) | baseline cross accuracy ≥ 0.6 in both directions |
| *after DA* (Fig. 2c / 2f) | post-DA cross accuracy ≥ 0.6 in both directions |
| *independent* (Fig. 3a / 3d) | *cross*, and in at least one direction DcdRatio > 1 with Cohen's d_acc in its top 25% |
| *alignable* (Fig. 3b / 3e) | baseline cross ≤ 0.6 both ways, baseline CovDist in its top 25%, DcdRatio > 1 and d_acc in its top 25% in both directions, CovDistRatio < 1 in both, and Cohen's d_dist below its 75th percentile in both |
| *dependent* (Fig. 3c / 3f) | *cross*, and DcdRatio < 1 in both directions with a low Cohen's d_acc |

The content and reality rules are written out separately because two details differ.
They are kept exactly as they were run for the paper's figures:

| Detail | Content | Reality |
|---|---|---|
| *alignable*: baseline CovDist in top 25% | in **both** directions | in **either** direction |
| *dependent*: "low" Cohen's d_acc | **below its 75th percentile** in both directions | **< 0** in both directions |

The dependent rule does not use the CovDist conditions listed in paper Tables 6–7.

In [ ]:
T = THR


def content_categories(m):
    ab_ok_base = (m["cross_ab"] >= T["cross"]) & (m["cross_ba"] >= T["cross"])
    ab_low_base = (m["cross_ab"] <= T["cross"]) & (m["cross_ba"] <= T["cross"])
    dz_hi_ab, dz_hi_ba = m["dz_ab"] > pct(m["dz_ab"]), m["dz_ba"] > pct(m["dz_ba"])
    gain_ab = (m["ratio_ab"] > T["dcd_ratio"]) & dz_hi_ab
    gain_ba = (m["ratio_ba"] > T["dcd_ratio"]) & dz_hi_ba
    return {
        # Table S1: content in at least one domain
        "content_ever": (m["within_a"] >= T["cd_p2p"]) | (m["within_b"] >= T["cd_i2i"]),
        # Figure 2a, Table S2: content in both domains
        "content_within": (m["within_a"] >= T["cd_p2p"]) & (m["within_b"] >= T["cd_i2i"]),
        # Figure 2b: content generalizes across domains (baseline)
        "content_cross": ab_ok_base,
        # Figure 2c: content decodable across domains after DA
        "content_da": (m["da_ab"] >= T["cross"]) & (m["da_ba"] >= T["cross"]),
        # Figure 3a, Table S4: reality-independent content signal
        "content_independent": ab_ok_base & (gain_ab | gain_ba),
        # Figure 3b, Table S3: reality-alignable content signal
        "content_alignable": (
            ab_low_base
            & (m["cov_ab"] >= pct(m["cov_ab"])) & (m["cov_ba"] >= pct(m["cov_ba"]))
            & gain_ab & gain_ba
            & (m["covratio_ab"] < T["covdist_ratio"]) & (m["covratio_ba"] < T["covdist_ratio"])
            & (m["covdz_ab"] < pct(m["covdz_ab"])) & (m["covdz_ba"] < pct(m["covdz_ba"]))
        ),
        # Figure 3c: reality-dependent content signal (DA makes it worse)
        "content_dependent": (
            ab_ok_base
            & (m["ratio_ab"] < T["dcd_ratio"]) & (m["ratio_ba"] < T["dcd_ratio"])
            & (m["dz_ab"] < pct(m["dz_ab"])) & (m["dz_ba"] < pct(m["dz_ba"]))
        ),
    }


def reality_categories(m):
    ab_ok_base = (m["cross_ab"] >= T["cross"]) & (m["cross_ba"] >= T["cross"])
    ab_low_base = (m["cross_ab"] <= T["cross"]) & (m["cross_ba"] <= T["cross"])
    gain_ab = (m["ratio_ab"] > T["dcd_ratio"]) & (m["dz_ab"] > pct(m["dz_ab"]))
    gain_ba = (m["ratio_ba"] > T["dcd_ratio"]) & (m["dz_ba"] > pct(m["dz_ba"]))
    return {
        # Table S5: reality in at least one content
        "reality_ever": (m["within_a"] >= T["rd_within"]) | (m["within_b"] >= T["rd_within"]),
        # Figure 2d, Table S6: reality in both contents
        "reality_within": (m["within_a"] >= T["rd_within"]) & (m["within_b"] >= T["rd_within"]),
        # Figure 2e: reality generalizes across contents (baseline)
        "reality_cross": ab_ok_base,
        # Figure 2f: reality decodable across contents after DA
        "reality_da": (m["da_ab"] >= T["cross"]) & (m["da_ba"] >= T["cross"]),
        # Figure 3d, Table S8: content-independent reality signal
        "reality_independent": ab_ok_base & (gain_ab | gain_ba),
        # Figure 3e, Table S7: content-alignable reality signal
        "reality_alignable": (
            ab_low_base
            & ((m["cov_ab"] >= pct(m["cov_ab"])) | (m["cov_ba"] >= pct(m["cov_ba"])))
            & gain_ab & gain_ba
            & (m["covratio_ab"] < T["covdist_ratio"]) & (m["covratio_ba"] < T["covdist_ratio"])
            & (m["covdz_ab"] < pct(m["covdz_ab"])) & (m["covdz_ba"] < pct(m["covdz_ba"]))
        ),
        # Figure 3f: content-dependent reality signal (DA makes it worse)
        "reality_dependent": (
            ab_ok_base
            & (m["ratio_ab"] < T["dcd_ratio"]) & (m["ratio_ba"] < T["dcd_ratio"])
            & (m["dz_ab"] < 0) & (m["dz_ba"] < 0)
        ),
    }

## Per-subject masks

Each subject gets one binary mask per category, saved under
`OUTPUT_ROOT/subject_masks/<subject>/`.

In [ ]:
masks = {}     # category -> list of (subject, mask)
ref_img = None

for analysis, rules in (("content", content_categories), ("reality", reality_categories)):
    for subject in SUBJECTS:
        img, m = load_subject(analysis, subject)
        if ref_img is None:
            ref_img = img
        elif img.shape != ref_img.shape or not np.allclose(img.affine, ref_img.affine):
            raise ValueError(f"Subject {subject} ({analysis}) is not on the common MNI grid")
        out_dir = OUTPUT_ROOT / "subject_masks" / subject
        out_dir.mkdir(parents=True, exist_ok=True)
        subject_masks = {k: np.asarray(v, dtype=bool) for k, v in rules(m).items()}
        for name, mask in subject_masks.items():
            masks.setdefault(name, []).append((subject, mask))
            new_img_like(ref_img, mask.astype(np.uint8)).to_filename(str(out_dir / f"{name}_mask.nii.gz"))
        print(f"{analysis:8s} subject {subject:>2s}: " + ", ".join(
            f"{k}={int(v.sum())}" for k, v in subject_masks.items()))

assert all(len(v) == len(SUBJECTS) for v in masks.values()), "Not every subject produced every mask"

## Group overlap maps

For each category, count how many subjects have each voxel, keep voxels found in more
than `OVERLAP_MIN_SUBJECTS` subjects, and express the count as a percentage of
subjects. Maps are saved under `OUTPUT_ROOT/overlap/`.

In [ ]:
overlap_dir = OUTPUT_ROOT / "overlap"
overlap_dir.mkdir(parents=True, exist_ok=True)
group = {}  # category -> dict(sum=..., keep=..., percent=...)
n_sub = len(SUBJECTS)

for name, entries in masks.items():
    total = np.sum([mk for _, mk in entries], axis=0).astype(float)
    keep = total > OVERLAP_MIN_SUBJECTS
    percent = np.where(keep, total / n_sub * 100.0, 0.0)
    group[name] = {"sum": total, "keep": keep, "percent": percent}
    new_img_like(ref_img, total).to_filename(str(overlap_dir / f"{name}_overlap_sum.nii.gz"))
    new_img_like(ref_img, keep.astype(np.uint8)).to_filename(str(overlap_dir / f"{name}_overlap_gt{OVERLAP_MIN_SUBJECTS}_mask.nii.gz"))
    new_img_like(ref_img, percent).to_filename(str(overlap_dir / f"{name}_overlap_gt{OVERLAP_MIN_SUBJECTS}_percent.nii.gz"))

summary = pd.DataFrame(
    [{"category": k, "voxels_in_group_map": int(v["keep"].sum()),
      "max_subjects_at_a_voxel": int(v["sum"].max())} for k, v in group.items()]
)
summary.to_csv(OUTPUT_ROOT / "group_voxel_counts.csv", index=False)
summary

## Figures 2 and 3

Each panel shows the group overlap map (left) and, per subject, the percentage of the
group-map voxels that the subject contributes (right).

In [ ]:
PANELS = {
    "Fig2a_content_within": ("content_within", "Content signal present within domain"),
    "Fig2b_content_cross": ("content_cross", "Content generalizes across domain"),
    "Fig2c_content_da": ("content_da", "Content-decodable after alignment"),
    "Fig2d_reality_within": ("reality_within", "Reality signal present within content"),
    "Fig2e_reality_cross": ("reality_cross", "Reality generalizes across content"),
    "Fig2f_reality_da": ("reality_da", "Reality-decodable after alignment"),
    "Fig3a_content_independent": ("content_independent", "Reality-independent content signal"),
    "Fig3b_content_alignable": ("content_alignable", "Reality-alignable content signal"),
    "Fig3c_content_dependent": ("content_dependent", "Reality-dependent content signal"),
    "Fig3d_reality_independent": ("reality_independent", "Content-independent reality signal"),
    "Fig3e_reality_alignable": ("reality_alignable", "Content-alignable reality signal"),
    "Fig3f_reality_dependent": ("reality_dependent", "Content-dependent reality signal"),
}
fig_dir = OUTPUT_ROOT / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)


def subject_contribution(name):
    keep = group[name]["keep"]
    n_keep = int(keep.sum())
    return pd.DataFrame([
        {"subject": int(s), "n_voxels": int((mk & keep).sum()),
         "pct_of_group_voxels": (mk & keep).sum() / n_keep * 100.0 if n_keep else 0.0}
        for s, mk in masks[name]
    ])


def draw_panel(name, title, out_png):
    df = subject_contribution(name)
    df.to_csv(out_png.with_suffix(".csv"), index=False)
    fig = plt.figure(figsize=(12, 4.6), facecolor="white")
    ax_brain = fig.add_axes([0.05, 0.22, 0.67, 0.62])
    plotting.plot_glass_brain(
        new_img_like(ref_img, group[name]["percent"]), display_mode=DISPLAY_MODE,
        cmap=OVERLAP_CMAP, black_bg=False, colorbar=False, vmin=0.0, vmax=100.0,
        threshold=0.5,  # leave voxels outside the group map blank (kept voxels are >= 27.8%)
        axes=ax_brain,
    )
    cax = fig.add_axes([0.235, 0.155, 0.30, 0.032])
    sm = cm.ScalarMappable(norm=colors.Normalize(0.0, 100.0), cmap=OVERLAP_CMAP)
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_label("% of subjects", fontsize=9)
    cbar.ax.tick_params(labelsize=8, pad=1)
    ax_bar = fig.add_axes([0.78, 0.22, 0.11, 0.56])
    ax_bar.barh(df["subject"], df["pct_of_group_voxels"], color="#5B6573",
                edgecolor="#3E4652", linewidth=0.6, height=0.72)
    ax_bar.set_xlim(0, 100)
    ax_bar.set_xticks([0, 50, 100])
    ax_bar.set_yticks(df["subject"])
    ax_bar.set_xlabel("% of overlap voxels")
    ax_bar.set_ylabel("Subject")
    ax_bar.grid(axis="y", alpha=0.25)
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close(fig)


for panel, (name, title) in PANELS.items():
    out_png = fig_dir / f"{panel}.png"
    draw_panel(name, title, out_png)
    print("saved", out_png)

## Tables S1–S8

Regions come from the Harvard–Oxford cortical atlas (max-probability, 25% threshold,
2 mm), resampled to the group-map grid with nearest-neighbour interpolation. Columns
match the paper:

- **Overlap voxel count**: group-map voxels in that region.
- **% of all overlap voxels**: that count as a share of all group-map voxels,
  including voxels outside any atlas label.
- **Subjects with overlap**: subjects whose own mask touches at least one
  group-map voxel in that region.

In [ ]:
TABLES = {
    "S1": ("content_ever", "Content decoding, represents content ever"),
    "S2": ("content_within", "Content decoding, represents content always"),
    "S3": ("content_alignable", "Content decoding, different but alignable content representations"),
    "S4": ("content_independent", "Content decoding, similar (and alignable) content representations"),
    "S5": ("reality_ever", "Reality decoding, represents reality-ness ever"),
    "S6": ("reality_within", "Reality decoding, represents reality always"),
    "S7": ("reality_alignable", "Reality decoding, different but alignable reality representations"),
    "S8": ("reality_independent", "Reality decoding, similar (and alignable) reality representations"),
}

atlas = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
atlas_data = np.rint(
    resample_to_img(load_img(atlas.maps), ref_img, interpolation="nearest",
                    force_resample=True, copy_header=True).get_fdata()
).astype(int)
atlas_labels = list(atlas.labels)

table_dir = OUTPUT_ROOT / "tables"
table_dir.mkdir(parents=True, exist_ok=True)


def region_table(name):
    keep = group[name]["keep"]
    n_keep = int(keep.sum())
    rows = []
    for label_id, count in zip(*np.unique(atlas_data[keep], return_counts=True)):
        if label_id == 0:
            continue
        region = (atlas_data == label_id) & keep
        n_subj = sum(bool(np.any(mk & region)) for _, mk in masks[name])
        rows.append({
            "Atlas Label ID": int(label_id),
            "Atlas Region": atlas_labels[label_id],
            "Overlap Voxel Count": int(count),
            "% of All Overlap Voxels": round(count / n_keep * 100.0, 2),
            "Subjects with Overlap": n_subj,
            "% Subjects with Overlap": round(n_subj / n_sub * 100.0, 2),
        })
    cols = ["Atlas Label ID", "Atlas Region", "Overlap Voxel Count", "% of All Overlap Voxels",
            "Subjects with Overlap", "% Subjects with Overlap"]
    return pd.DataFrame(rows, columns=cols).sort_values("Overlap Voxel Count", ascending=False, ignore_index=True)


tables = {}
for tab, (name, title) in TABLES.items():
    df = region_table(name)
    tables[tab] = df
    df.to_csv(table_dir / f"Table_{tab}_{name}.csv", index=False)
    print(f"\nTable {tab}: {title}  ({int(group[name]['keep'].sum())} group-map voxels)")
    print(df.head(10).to_string(index=False))